# Gate fusion in `QarpSimulator`

Every kernel call is one pass over all $2^n$ amplitudes, so a statevector
run costs *passes × state size*.  Before dispatching, `QarpSimulator`
fuses runs of gates into dense blocks of up to `fusion_max_qubits` qubits
and applies each block in a single pass — the same idea as qiskit-aer's
`fusion_enable`, and the reason Aer used to win above 16 qubits.

The pass is exact (global phase included) and simulator-internal: the
transpiler's `optimize` output is unchanged, and `unitary_matrix`, the
adjoint gradient and noisy trajectories still run gate by gate.

## The knob

`0` = raw per-gate dispatch, `1` = single-qubit fusion only, `k ≥ 2` = dense blocks.  `QARP_FUSION_MAX_QUBITS` sets the process-wide default for every simulator an engine constructs.

In [ ]:
import qarpx as qx

sim = qx.QarpSimulator()
print("default:", sim.fusion_max_qubits, "  max:", qx.QarpSimulator.MAX_FUSION_QUBITS)

## Same state, fewer passes

A 16-qubit brickwork circuit; every width gives the same amplitudes as the unfused `unitary_matrix` oracle would, only faster.

In [ ]:
import time

import numpy as np

from qarp.blocks import SimpleBlock

n, layers = 16, 6
rng = np.random.default_rng(7)
circuit = SimpleBlock(n)
for layer in range(layers):
    for a in range(layer % 2, n - 1, 2):
        circuit.rz(a, rng.uniform()).ry(a, rng.uniform()).rz(a, rng.uniform())
        circuit.rz(a + 1, rng.uniform()).ry(a + 1, rng.uniform()).rz(a + 1, rng.uniform())
        circuit.cx(a, a + 1).ry(a, rng.uniform()).ry(a + 1, rng.uniform()).cx(a, a + 1)
circuit.build()
cmds = circuit.flatten()


def timed(width, reps=5):
    sim = qx.QarpSimulator()
    sim.fusion_max_qubits = width
    sim.statevector(cmds, n)  # warm
    t = time.perf_counter()
    for _ in range(reps):
        psi = sim.statevector(cmds, n)
    return np.asarray(psi), (time.perf_counter() - t) / reps


reference, t_raw = timed(0)
for width in (1, 2, 3, 4):
    psi, t = timed(width)
    print(f"k={width}: {t * 1e3:7.1f} ms  ({t_raw / t:4.1f}x)   max |Δψ| = {np.abs(psi - reference).max():.1e}")

Wider is not always better: a $k$-qubit block costs $2^k$ multiplies per
amplitude, so past the memory-bound regime the extra arithmetic shows.
The built-in default is the width measured fastest on this kernel across
the statevector benchmark families; leave it unless you are measuring.